# Face detector - train / validate / TEST on a 70-20-10 split

Re-runs the YOLOv8n vs YOLO11n vs RT-DETR-l comparison on a proper three-way split, so the
reported accuracy comes from images the model never saw during training **or** model selection.

| Split | Images (faces) | Used for |
|---|---|---|
| train (70%) | 2,809 (7,483) | fitting the weights |
| val (20%) | 803 (2,008) | early stopping + picking `best.pt` (seen indirectly - **not** reported as final accuracy) |
| test (10%) | 402 (1,005) | evaluated **once**, after training - **these are the numbers for the paper** |

Built locally by `scripts/split_face_dataset.py --zip` (seed 42, stratified by source/WIDER event, exact
duplicates removed). See `datasets/face_split_70_20_10/split_report.md`.

It also produces Keras-style training graphs (training vs validation **accuracy** and **loss** per epoch)
plus the paper's figure set, all saved as PNGs to Drive.

**Before running**
1. Upload `datasets/face_split_70_20_10.zip` to Google Drive root (`MyDrive/face_split_70_20_10.zip`).
2. Runtime -> Change runtime type -> **T4 GPU**.
3. Run all. If a session runs out before all three models finish, set `RUN` in the config cell to the
   models still missing and run again - finished models are already saved to Drive.

**Rule:** do not change any setting after looking at test results and then re-test. The test set is
only honest if it is used once.

In [ ]:
# Pinned so the run is reproducible and matches the evaluation code below.
!pip install -q ultralytics==8.4.155
import ultralytics
ultralytics.checks()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, yaml

DATASET_ZIP = '/content/drive/MyDrive/face_split_70_20_10.zip'
DATA_ROOT = '/content/data'
DATA_DIR = f'{DATA_ROOT}/face_split_70_20_10'

!mkdir -p {DATA_ROOT}
!unzip -q -o "{DATASET_ZIP}" -d {DATA_ROOT}

# Point the yaml at this Colab runtime (the local copy points at the Windows machine).
yaml_path = f'{DATA_DIR}/face_split.yaml'
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = DATA_DIR
with open(yaml_path, 'w') as f:
    yaml.safe_dump(cfg, f)

counts = {s: len(os.listdir(f'{DATA_DIR}/images/{s}')) for s in ('train', 'val', 'test')}
print(cfg)
print(counts)
assert counts == {'train': 2809, 'val': 803, 'test': 402}, 'Unexpected split sizes - wrong zip?'

In [ ]:
# ---- Config: identical for every model so the comparison is fair ----
EPOCHS = 100
IMGSZ = 640
PATIENCE = 20     # early stopping on the VAL split
SEED = 42

# (name, pretrained weights, batch). RT-DETR is heavier -> smaller batch for the 16 GB T4.
MODELS = [
    ('yolov8n', 'yolov8n.pt', 16),
    ('yolo11n', 'yolo11n.pt', 16),
    ('rtdetr-l', 'rtdetr-l.pt', 8),
]
RUN = ['yolov8n', 'yolo11n', 'rtdetr-l']   # trim this to resume in a new session

# Keras-style "training accuracy": YOLO/RT-DETR never score the training set, so after every epoch
# last.pt is evaluated on this many fixed, un-augmented training images (same metric code as val).
# 800 ~= the val split size. Costs roughly +15 s/epoch for the nano models, more for RT-DETR.
TRAIN_EVAL_N = 800

REALTIME_FPS = 30   # threshold line on the accuracy-vs-speed chart (paper Fig. 49)

OUT_DIR = '/content/drive/MyDrive/knowing-eye-70-20-10'   # everything that must survive the session
FIG_DIR = f'{OUT_DIR}/figures'
RUNS = '/content/runs'
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
import csv, math, random, shutil
from pathlib import Path
from ultralytics import YOLO, RTDETR


def model_class(name):
    return RTDETR if name.startswith('rtdetr') else YOLO


# Fixed training subset for the per-epoch "training accuracy" pass.
train_images = sorted(str(p) for p in Path(DATA_DIR, 'images', 'train').iterdir())
random.Random(SEED).shuffle(train_images)
Path(DATA_DIR, 'train_eval.txt').write_text('\n'.join(train_images[:TRAIN_EVAL_N]) + '\n')
train_eval_yaml = f'{DATA_DIR}/train_eval.yaml'
with open(train_eval_yaml, 'w') as f:
    yaml.safe_dump({'path': DATA_DIR, 'train': 'images/train', 'val': f'{DATA_DIR}/train_eval.txt',
                    'nc': 1, 'names': ['face']}, f)

TRAIN_EVAL_FIELDS = ['epoch', 'precision', 'recall', 'map50', 'map']


def train_eval_callback(name):
    out_csv = Path(OUT_DIR, name, 'train_eval_per_epoch.csv')
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    out_csv.write_text(','.join(TRAIN_EVAL_FIELDS) + '\n')   # fresh file per training run
    logged = set()

    def on_fit_epoch_end(trainer):
        # Label by the epoch just written to results.csv; Ultralytics also fires this hook once more
        # during its final evaluation, which would otherwise log a phantom extra epoch.
        lines = Path(trainer.csv).read_text().strip().splitlines() if Path(trainer.csv).exists() else []
        if len(lines) < 2:
            return
        epoch = int(float(lines[-1].split(',')[0]))
        if epoch in logged:
            return
        logged.add(epoch)
        row = {'epoch': epoch}
        try:
            m = model_class(name)(str(trainer.last)).val(
                data=train_eval_yaml, split='val', imgsz=IMGSZ, batch=16, device=str(trainer.device),
                plots=False, verbose=False, project=RUNS, name=f'{name}-train-eval', exist_ok=True)
            row.update(precision=m.box.mp, recall=m.box.mr, map50=m.box.map50, map=m.box.map)
        except Exception as e:   # e.g. NaN weights after RT-DETR diverges
            print(f'[{name}] train-eval failed at epoch {row["epoch"]}: {e}')
            row.update(precision=math.nan, recall=math.nan, map50=math.nan, map=math.nan)
        with out_csv.open('a', newline='') as f:
            csv.DictWriter(f, TRAIN_EVAL_FIELDS).writerow(row)

    return on_fit_epoch_end


for name, weights, batch in MODELS:
    if name not in RUN:
        continue
    print(f'\n{"=" * 60}\nTRAINING {name}  (epochs={EPOCHS}, batch={batch})\n{"=" * 60}')
    model = model_class(name)(weights)
    model.add_callback('on_fit_epoch_end', train_eval_callback(name))
    try:
        model.train(
            data=yaml_path, epochs=EPOCHS, imgsz=IMGSZ, batch=batch, patience=PATIENCE,
            seed=SEED, deterministic=True, single_cls=True,
            project=RUNS, name=name, exist_ok=True,
        )
    except Exception as e:   # e.g. RT-DETR diverging - keep whatever best.pt was saved
        print(f'[{name}] training stopped with: {e}')

    save_dir = Path(model.trainer.save_dir)
    dest = Path(OUT_DIR, name)
    for f in ('weights/best.pt', 'results.csv', 'results.png', 'args.yaml'):
        if (save_dir / f).exists():
            shutil.copy2(save_dir / f, dest / Path(f).name)
    print(f'[{name}] saved to {dest}')

## Final evaluation - TEST split (run once)

For each model this loads `best.pt` and evaluates it on the **test** split. It also evaluates on the val
split for reference only (that is the selection set, so it runs optimistic - do not report it as accuracy).

Metric definitions (single class, so there are no true negatives):
- **Precision** = TP / (TP + FP), **Recall** = TP / (TP + FN), **F1** = 2PR / (P + R)
- **Accuracy** = TP / (TP + FP + FN) = PR / (P + R − PR) - the same quantity the paper's "Accuracy" column
  already used (it equals F1 / (2 − F1)); the paper must define it this way.
- **mAP@0.5 / mAP@0.5:0.95** - Ultralytics/COCO average precision.
- **TP / FP / FN counts** - from the confusion matrix at conf 0.25, IoU 0.45 (for a confusion-matrix figure).
- **Inference (ms)** - measured on this Colab **T4 GPU**; say so in the paper.

In [ ]:
import numpy as np
import pandas as pd


def evaluate(name, split):
    model = model_class(name)(f'{OUT_DIR}/{name}/best.pt')
    m = model.val(data=yaml_path, split=split, imgsz=IMGSZ, batch=16, plots=True,
                  project=RUNS, name=f'{name}-{split}', exist_ok=True)
    p, r = float(m.box.mp), float(m.box.mr)
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    acc = p * r / (p + r - p * r) if p + r else 0.0
    cm = m.confusion_matrix.matrix          # rows = predicted, cols = actual; index 1 = background
    inf_ms = float(m.speed['inference'])
    row = {
        'model': name, 'split': split,
        'Accuracy (%)': round(acc * 100, 2),
        'Precision (%)': round(p * 100, 2),
        'Recall (%)': round(r * 100, 2),
        'F1 (%)': round(f1 * 100, 2),
        'mAP50 (%)': round(m.box.map50 * 100, 2),
        'mAP50-95 (%)': round(m.box.map * 100, 2),
        'TP': int(cm[0, 0]), 'FP': int(cm[0, 1]), 'FN': int(cm[1, 0]),
        'Inference T4 (ms)': round(inf_ms, 2),
        'FPS T4': round(1000 / inf_ms, 1) if inf_ms else float('nan'),
        'Params (M)': round(sum(x.numel() for x in model.model.parameters()) / 1e6, 2),
    }
    if split == 'test':
        # Mean confidence of surviving boxes at the standard conf >= 0.25 threshold (paper's Table 2 column).
        confs = []
        for res in model.predict(f'{DATA_DIR}/images/test', conf=0.25, imgsz=IMGSZ, stream=True, verbose=False):
            confs.extend(res.boxes.conf.cpu().numpy().tolist())
        row['Mean Confidence (%)'] = round(float(np.mean(confs)) * 100, 2) if confs else 0.0
        row['Detections (n)'] = len(confs)
        dest = Path(OUT_DIR, name, 'test_plots')
        dest.mkdir(parents=True, exist_ok=True)
        for png in Path(m.save_dir).glob('*.png'):
            shutil.copy2(png, dest / png.name)
    return row


done = [n for n, _, _ in MODELS if Path(OUT_DIR, n, 'best.pt').exists()]
test_rows = [evaluate(n, 'test') for n in done]
val_rows = [evaluate(n, 'val') for n in done]

test_df = pd.DataFrame(test_rows)
val_df = pd.DataFrame(val_rows)
test_df.to_csv(f'{OUT_DIR}/test_results_70_20_10.csv', index=False)
val_df.to_csv(f'{OUT_DIR}/val_results_70_20_10_reference_only.csv', index=False)
print('TEST split - report these:')
test_df

## Training history + figures

**How to read the Keras-style curves (put this in the paper's figure discussion):**
- *Training accuracy / metrics* are measured after every epoch on 800 fixed **un-augmented** training images,
  with exactly the same evaluation code as validation - so the train-vs-val gap is a fair overfitting check.
- *Training loss* is what Ultralytics logs during the training pass itself, i.e. on **augmented** batches
  (mosaic, flips, colour jitter). Validation loss is on clean images. That is why validation loss can sit
  *below* training loss, and why training loss drops sharply ~10 epochs before the end
  (`close_mosaic=10` switches mosaic augmentation off) - neither is a bug.
- Accuracy = TP / (TP + FP + FN) at each epoch's best-F1 operating point (see the metric definitions above).
- The dotted line marks the epoch kept as `best.pt` (highest validation mAP@0.5:0.95).
- Checkpoint tables/charts are **validation** numbers; the bar charts and confusion matrix are **test** numbers.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Palette and look matched to the paper's existing figures.
METRIC_COLORS = {
    'mAP@0.5': '#1F4E8C', 'Precision': '#2E8B57', 'F1-Score': '#D98C00', 'Recall': '#C0504D',
    'mAP@0.5:0.95': '#6A3D9A', 'Accuracy': '#2B8C8C',
}
MODEL_COLORS = {'yolov8n': '#1F4E8C', 'yolo11n': '#2E8B57', 'rtdetr-l': '#D98C00'}
DISPLAY = {'yolov8n': 'YOLOv8n', 'yolo11n': 'YOLOv11n', 'rtdetr-l': 'RT-DETR-l'}
TRAIN_C, VAL_C = '#1F4E8C', '#D98C00'

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 11, 'axes.titleweight': 'bold', 'axes.titlesize': 13,
    'axes.labelweight': 'bold', 'axes.labelsize': 12, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
})


def style(ax, xlabel=None, ylabel=None, title=None):
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_linewidth(1.6)
    ax.grid(axis='y', color='#DDDDDD', linewidth=1)
    ax.set_axisbelow(True)
    if xlabel:
        ax.set_xlabel(xlabel)
        if 'Epoch' in xlabel:
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    if ylabel:
        ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)


def legend_top(ax, ncol):
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.01), ncol=ncol, frameon=False,
              prop={'weight': 'bold', 'size': 10})


def mark_best(ax, best):
    ax.axvline(best, color='#5B7A99', linestyle=':', linewidth=1.5)
    ax.annotate(f'best: epoch {best}', xy=(best, 1), xycoords=('data', 'axes fraction'), xytext=(0, 4),
                textcoords='offset points', ha='center', va='bottom', color='#1F4E8C', fontsize=9,
                fontweight='bold')


def save(fig, fname):
    fig.savefig(f'{FIG_DIR}/{fname}', dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)


def f1_acc(p, r):
    s = p + r
    f1 = (2 * p * r / s).where(s > 0, 0.0)
    acc = (p * r / (s - p * r)).where(s > 0, 0.0)
    return f1, acc


def load_history(name):
    # One row per epoch: validation metrics + losses (results.csv) + training metrics (train-eval pass), in %.
    df = pd.read_csv(f'{OUT_DIR}/{name}/results.csv')
    df.columns = [c.strip() for c in df.columns]
    h = pd.DataFrame({'epoch': df['epoch'].astype(int)})
    p, r = df['metrics/precision(B)'], df['metrics/recall(B)']
    f1, acc = f1_acc(p, r)
    h['val_precision'], h['val_recall'] = p * 100, r * 100
    h['val_f1'], h['val_accuracy'] = f1 * 100, acc * 100
    h['val_map50'], h['val_map'] = df['metrics/mAP50(B)'] * 100, df['metrics/mAP50-95(B)'] * 100

    # Loss components differ by family: YOLO = box/cls/dfl, RT-DETR = giou/cls/l1.
    comps = [c.split('/', 1)[1] for c in df.columns if c.startswith('train/') and c.endswith('loss')]
    for comp in comps:
        h[f'train_{comp}'] = df[f'train/{comp}']
        if f'val/{comp}' in df:
            h[f'val_{comp}'] = df[f'val/{comp}']
    h['train_loss'] = h[[f'train_{c}' for c in comps]].sum(axis=1, min_count=len(comps))
    val_cols = [f'val_{c}' for c in comps if f'val_{c}' in h]
    h['val_loss'] = h[val_cols].sum(axis=1, min_count=len(val_cols)) if val_cols else math.nan

    te_path = Path(OUT_DIR, name, 'train_eval_per_epoch.csv')
    if te_path.exists():
        te = pd.read_csv(te_path).drop_duplicates('epoch', keep='last')
        tf1, tacc = f1_acc(te['precision'], te['recall'])
        te = pd.DataFrame({'epoch': te['epoch'].astype(int),
                           'train_precision': te['precision'] * 100, 'train_recall': te['recall'] * 100,
                           'train_f1': tf1 * 100, 'train_accuracy': tacc * 100,
                           'train_map50': te['map50'] * 100, 'train_map': te['map'] * 100})
        h = h.merge(te, on='epoch', how='left')
    best = int(h.loc[h['val_map'].fillna(-1).idxmax(), 'epoch'])   # how best.pt is chosen in 8.4
    return h, comps, best


histories = {n: load_history(n) for n in done}

In [ ]:
# Checkpoint tables (epochs 20/40/60/80/100 + best + last) - VALIDATION metrics with training accuracy/loss
# alongside, Keras-style. These replace the paper's Tables 3-5.
def history_table(name):
    h, _, best = histories[name]
    keep = sorted({e for e in (20, 40, 60, 80, 100) if e in set(h['epoch'])} | {best, int(h['epoch'].max())})
    cols = {
        'epoch': 'Epoch', 'val_accuracy': 'Accuracy (%)', 'val_recall': 'Recall (%)',
        'val_precision': 'Precision (%)', 'val_f1': 'F1-Score (%)', 'val_map50': 'mAP@0.5 (%)',
        'val_map': 'mAP@0.5:0.95 (%)', 'train_accuracy': 'Train Accuracy (%)',
        'train_loss': 'Train Loss', 'val_loss': 'Val Loss',
    }
    table = h[h['epoch'].isin(keep)][[c for c in cols if c in h]].rename(columns=cols).round(2)
    table['Note'] = table['Epoch'].map(lambda e: 'best.pt' if e == best else '')
    h.round(4).to_csv(f'{OUT_DIR}/{name}/history_all_epochs.csv', index=False)
    table.to_csv(f'{OUT_DIR}/{name}/history_checkpoints.csv', index=False)
    return table


for name in done:
    print(f'\n{DISPLAY[name]}  (validation metrics; train accuracy/loss for comparison)')
    display(history_table(name))

In [ ]:
# ---------- Per-model figures ----------
def fig_keras(name):
    # The classic Keras pair: model accuracy and model loss, training vs validation.
    h, _, best = histories[name]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    if 'train_accuracy' in h:
        a1.plot(h['epoch'], h['train_accuracy'], color=TRAIN_C, lw=2.2, label='Training')
    a1.plot(h['epoch'], h['val_accuracy'], color=VAL_C, lw=2.2, label='Validation')
    style(a1, 'Epoch', 'Accuracy (%)', 'Model Accuracy')
    a1.set_ylim(0, 100)
    mark_best(a1, best)
    a1.legend(loc='lower right', frameon=False, prop={'weight': 'bold'})
    a2.plot(h['epoch'], h['train_loss'], color=TRAIN_C, lw=2.2, label='Training')
    a2.plot(h['epoch'], h['val_loss'], color=VAL_C, lw=2.2, label='Validation')
    style(a2, 'Epoch', 'Loss (sum of components)', 'Model Loss')
    a2.set_ylim(bottom=0)
    mark_best(a2, best)
    a2.legend(loc='best', frameon=False, prop={'weight': 'bold'})
    fig.suptitle(f'{DISPLAY[name]} - training vs validation', fontweight='bold', fontsize=14, y=1.04)
    save(fig, f'{name}_keras_accuracy_loss.png')


def fig_train_vs_val_metrics(name):
    h, _, best = histories[name]
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    for ax, (key, label) in zip(axes.flat, [('precision', 'Precision'), ('recall', 'Recall'),
                                             ('f1', 'F1-Score'), ('map50', 'mAP@0.5')]):
        if f'train_{key}' in h:
            ax.plot(h['epoch'], h[f'train_{key}'], color=TRAIN_C, lw=2, label='Training')
        ax.plot(h['epoch'], h[f'val_{key}'], color=VAL_C, lw=2, label='Validation')
        style(ax, 'Epoch' if ax in axes[1] else None, f'{label} (%)', label)
        ax.set_ylim(0, 100)
        mark_best(ax, best)
        ax.legend(loc='lower right', frameon=False, prop={'weight': 'bold'})
    fig.suptitle(f'{DISPLAY[name]} - training vs validation metrics per epoch', fontweight='bold',
                 fontsize=14, y=1.01)
    fig.tight_layout()
    save(fig, f'{name}_train_vs_val_metrics.png')


def fig_loss_components(name):
    h, comps, best = histories[name]
    colors = ['#D98C00', '#C0504D', '#6A3D9A']
    fig, ax = plt.subplots(figsize=(13, 5.5))
    for comp, c in zip(comps, colors):
        short = comp.replace('_loss', '')
        label = {'box': 'Box', 'cls': 'Class', 'dfl': 'DFL', 'giou': 'GIoU', 'l1': 'L1'}.get(short, short)
        ax.plot(h['epoch'], h[f'train_{comp}'], color=c, lw=2.2, label=f'{label} Loss (train)')
        if f'val_{comp}' in h:
            ax.plot(h['epoch'], h[f'val_{comp}'], color=c, lw=2, ls='--', label=f'{label} Loss (val)')
    style(ax, 'Training Epoch', 'Loss')
    ax.set_ylim(bottom=0)
    mark_best(ax, best)
    legend_top(ax, ncol=len(comps))
    save(fig, f'{name}_loss_components.png')


def fig_val_metrics_all_epochs(name):
    # Same layout as the paper's "validation Precision, Recall, mAP@0.5, mAP@0.5:0.95 across all epochs".
    h, _, best = histories[name]
    fig, ax = plt.subplots(figsize=(13, 5.5))
    for key, label in [('val_map50', 'mAP@0.5'), ('val_map', 'mAP@0.5:0.95'),
                       ('val_precision', 'Precision'), ('val_recall', 'Recall')]:
        ax.plot(h['epoch'], h[key], color=METRIC_COLORS[label], lw=2.2, label=label)
    style(ax, 'Training Epoch', 'Score (%)')
    ax.set_ylim(0, 100)
    ax.set_xlim(1, h['epoch'].max())
    mark_best(ax, best)
    legend_top(ax, ncol=4)
    save(fig, f'{name}_val_metrics_all_epochs.png')


def fig_checkpoints(name):
    # Same layout as the paper's "performance metrics as a function of training epochs" (Fig. 47).
    h, _, _ = histories[name]
    pts = h[h['epoch'].isin([20, 40, 60, 80, 100])]
    if pts.empty:
        pts = h
    fig, ax = plt.subplots(figsize=(13, 6))
    for key, label in [('val_map50', 'mAP@0.5'), ('val_precision', 'Precision'), ('val_f1', 'F1-Score'),
                       ('val_recall', 'Recall'), ('val_accuracy', 'Accuracy')]:
        c = METRIC_COLORS[label]
        ax.plot(pts['epoch'], pts[key], color=c, lw=2.5, marker='o', ms=7, label=label)
        ax.annotate(f'{pts[key].iloc[-1]:.2f}', (pts['epoch'].iloc[-1], pts[key].iloc[-1]), xytext=(8, 0),
                    textcoords='offset points', va='center', color=c, fontweight='bold')
    style(ax, 'Training Epoch', 'Score (%)')
    ax.set_xticks(pts['epoch'])
    legend_top(ax, ncol=5)
    save(fig, f'{name}_checkpoint_metrics.png')


def fig_test_bars(name):
    # Same layout as the paper's "final detection performance" bar chart - but on the TEST split.
    row = test_df.set_index('model').loc[name]
    items = [('Accuracy', row['Accuracy (%)']), ('mAP@0.5', row['mAP50 (%)']), ('Recall', row['Recall (%)']),
             ('Precision', row['Precision (%)']), ('F1-Score', row['F1 (%)'])]
    fig, ax = plt.subplots(figsize=(13, 5.5))
    for i, (label, v) in enumerate(items):
        c = METRIC_COLORS[label]
        ax.bar(i, v, width=0.45, color=c, edgecolor='black', linewidth=1.2)
        ax.text(i, v + 1.5, f'{v:.2f}', ha='center', va='bottom', color=c, fontweight='bold', fontsize=12)
    ax.set_xticks(range(len(items)), [l for l, _ in items], fontweight='bold')
    style(ax, None, 'Score (%)', f'{DISPLAY[name]} - test-set detection performance (n = 402 images)')
    ax.set_ylim(0, 105)
    save(fig, f'{name}_test_metrics_bar.png')


def fig_confusion(name):
    row = test_df.set_index('model').loc[name]
    tp, fp, fn = int(row['TP']), int(row['FP']), int(row['FN'])
    grid = np.array([[tp, fn], [fp, np.nan]])
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    shade = np.nan_to_num(grid / max(tp + fp + fn, 1))
    ax.imshow(shade, cmap='Blues', vmin=0, vmax=1)
    labels = [[f'TP\n{tp}', f'FN\n{fn}'], [f'FP\n{fp}', 'TN\nn/a*']]
    for i in range(2):
        for j in range(2):
            dark = shade[i, j] > 0.5   # white text only on dark cells
            ax.text(j, i, labels[i][j], ha='center', va='center', fontsize=15, fontweight='bold',
                    color='white' if dark else '#1A1A1A')
    ax.set_xticks([0, 1], ['Face', 'Background'])
    ax.set_yticks([0, 1], ['Face', 'Background'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{DISPLAY[name]} - test-set confusion matrix')
    fig.text(0.5, -0.02, '*Detection has no true negatives (every non-face region would count). '
             'conf >= 0.25, IoU >= 0.45.', ha='center', fontsize=8.5, color='#555555')
    save(fig, f'{name}_test_confusion_matrix.png')


for name in done:
    print(f'\n######## {DISPLAY[name]} ########')
    fig_keras(name)
    fig_train_vs_val_metrics(name)
    fig_loss_components(name)
    fig_val_metrics_all_epochs(name)
    fig_checkpoints(name)
    fig_test_bars(name)
    fig_confusion(name)

In [ ]:
# ---------- Model comparison figures (TEST split) ----------
def fig_compare_bars():
    metrics = [('mAP@0.5', 'mAP50 (%)'), ('mAP@0.5:0.95', 'mAP50-95 (%)'), ('Precision', 'Precision (%)'),
               ('Recall', 'Recall (%)'), ('F1-Score', 'F1 (%)'), ('Accuracy', 'Accuracy (%)')]
    t = test_df.set_index('model')
    width = 0.8 / len(done)
    fig, ax = plt.subplots(figsize=(14, 6))
    for k, name in enumerate(done):
        xs = np.arange(len(metrics)) + (k - (len(done) - 1) / 2) * width
        vals = [t.loc[name, col] for _, col in metrics]
        ax.bar(xs, vals, width * 0.92, color=MODEL_COLORS[name], edgecolor='black', linewidth=1.1,
               label=DISPLAY[name])
        for x, v in zip(xs, vals):
            ax.text(x, v + 0.8, f'{v:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_xticks(range(len(metrics)), [m for m, _ in metrics], fontweight='bold')
    style(ax, None, 'Score (%)')
    ax.set_ylim(0, 105)
    legend_top(ax, ncol=len(done))
    save(fig, 'compare_test_metrics.png')


def fig_compare_speed():
    # Accuracy-vs-speed bubble chart (paper Fig. 49). Speed = Colab T4 GPU.
    t = test_df.set_index('model')
    fig, ax = plt.subplots(figsize=(13, 6))
    xmax = max(t['FPS T4'].max() * 1.12, REALTIME_FPS * 2)
    ax.axvspan(0, REALTIME_FPS, color='#F2D7D7', alpha=0.8, zorder=0)
    ax.axvline(REALTIME_FPS, color='#C0504D', ls='--', lw=1.5)
    ax.text(REALTIME_FPS + xmax * 0.01, 0.97, f'{REALTIME_FPS} FPS - live-webcam real-time threshold',
            transform=ax.get_xaxis_transform(), color='#C0504D', fontweight='bold', va='top')
    # Labels alternate above/below (sorted by speed) so neighbouring models never overlap.
    for k, name in enumerate(sorted(done, key=lambda n: t.loc[n, 'FPS T4'])):
        x, y, params = t.loc[name, 'FPS T4'], t.loc[name, 'mAP50 (%)'], t.loc[name, 'Params (M)']
        c = MODEL_COLORS[name]
        ax.scatter(x, y, s=params * 120, color=c, alpha=0.35, edgecolor=c, linewidth=2.5, zorder=3)
        ax.scatter(x, y, s=18, color=c, zorder=4)
        above = k % 2 == 0
        side = 'left' if x < xmax * 0.15 else 'right' if x > xmax * 0.85 else 'center'
        dx = {'left': 30, 'right': -12, 'center': 0}[side]
        ax.annotate(f'{DISPLAY[name]}\n{y:.2f} mAP · {x:.0f} FPS · {params:.1f}M', (x, y),
                    xytext=(dx, 26 if above else -30), textcoords='offset points', ha=side,
                    va='bottom' if above else 'top', color=c, fontweight='bold', fontsize=10, zorder=5)
    lo, hi = t['mAP50 (%)'].min(), t['mAP50 (%)'].max()
    ax.set_ylim(lo - 3, hi + 2.5)
    ax.set_xlim(0, xmax)
    style(ax, 'Inference speed on T4 GPU (FPS) -> faster', 'mAP@0.5 on test split (%)')
    ax.text(0.99, 0.02, 'bubble area ∝ model size (params)', transform=ax.transAxes, ha='right',
            fontsize=9, style='italic', color='#666666')
    save(fig, 'compare_accuracy_vs_speed.png')


def fig_compare_curves():
    # All models on one Keras-style pair: dashed = training, solid = validation.
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    for name in done:
        h, _, _ = histories[name]
        c = MODEL_COLORS[name]
        if 'train_accuracy' in h:
            a1.plot(h['epoch'], h['train_accuracy'], color=c, lw=1.6, ls='--', alpha=0.85)
        a1.plot(h['epoch'], h['val_accuracy'], color=c, lw=2.2, label=DISPLAY[name])
        a2.plot(h['epoch'], h['train_loss'], color=c, lw=1.6, ls='--', alpha=0.85)
        a2.plot(h['epoch'], h['val_loss'], color=c, lw=2.2, label=DISPLAY[name])
    style(a1, 'Epoch', 'Accuracy (%)', 'Model Accuracy (solid = validation, dashed = training)')
    a1.set_ylim(0, 100)
    style(a2, 'Epoch', 'Loss', 'Model Loss (solid = validation, dashed = training)')
    a2.set_ylim(bottom=0)
    a1.legend(loc='lower right', frameon=False, prop={'weight': 'bold'})
    a2.legend(loc='upper right', frameon=False, prop={'weight': 'bold'})
    save(fig, 'compare_keras_accuracy_loss.png')


fig_compare_bars()
fig_compare_speed()
fig_compare_curves()
print('Figures saved to', FIG_DIR)
print(sorted(os.listdir(FIG_DIR)))

## Download

Everything is in `MyDrive/knowing-eye-70-20-10/`:
- `test_results_70_20_10.csv` - **final accuracy table** (replaces the paper's Tables 1-2 numbers)
- `figures/` - every graph as a 200-dpi PNG:
  - `<model>_keras_accuracy_loss.png` - Keras-style model accuracy + model loss, training vs validation
  - `<model>_train_vs_val_metrics.png` - training vs validation Precision / Recall / F1 / mAP@0.5
  - `<model>_loss_components.png` - each loss term, training (solid) vs validation (dashed)
  - `<model>_val_metrics_all_epochs.png` - validation P / R / mAP per epoch (paper Figs. 50/52/54)
  - `<model>_checkpoint_metrics.png` - checkpoint epochs 20-100 (paper Fig. 47)
  - `<model>_test_metrics_bar.png` - final TEST metrics (paper Fig. 48)
  - `<model>_test_confusion_matrix.png` - TEST confusion matrix
  - `compare_test_metrics.png`, `compare_accuracy_vs_speed.png`, `compare_keras_accuracy_loss.png`
- `<model>/history_checkpoints.csv`, `history_all_epochs.csv` - training-history tables (paper Tables 3-5)
- `<model>/test_plots/` - Ultralytics' own PR / P / R / F1 curves and confusion matrix on the test split
- `<model>/best.pt` - the weights the test numbers belong to
- `val_results_70_20_10_reference_only.csv` - selection-set numbers, for the appendix at most